# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushdevo/10x.ai/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# Method Choice and Why

I selected a Gradient Boosted Decision Tree model (LightGBM/CatBoost/XGBoost).

Why it fits this problem:

- The dataset contains mostly tabular features such as impressions, clicks, CTR, content age, word count, competition's levels, and trend indicators.
- Tree-based models handle non-linear relationships better than a simple rule-based baseline.
- They work well with mixed feature scales and missing values.
- Feature importance can be inspected, making the model easier to explain.

My goal is not to predict exact future traffic but to identify content that is likely declining and should be refreshed. This makes gradient boosting a good fit because it captures complex interactions while remaining interpretable.# Method Choice and Why

I selected a Gradient Boosted Decision Tree model (LightGBM/CatBoost/XGBoost).

Why it fits this problem:

- The dataset contains mostly tabular features such as impressions, clicks, CTR, content age, word count, competition level, and trend indicators.
- Tree-based models handle non-linear relationships better than a simple rule-based baseline.
- They work well with mixed feature scales and missing values.
- Feature importance can be inspected, making the model easier to explain.

My goal is not to predict exact future traffic but to identify content that is likely declining and should be refreshed. This makes gradient boosting a good fit because it captures complex interactions while remaining interpretable.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# Split Design

I used GroupShuffleSplit with client_id as the grouping variable.

Why this is an honest split:

- Multiple pages belong to the same client.
- Random row-level splitting would allow information leakage between train and test sets.
- Grouping by client ensures the model is evaluated on clients it has not seen during training.

This better reflects real-world deployment where the model must generalize to new client content rather than memorizing patterns from the same client.

## 3. Training + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.metrics import f1_score

baseline_pred = (test_df["days_since_last_update"] > 180).astype(int)

baseline_f1 = f1_score(
    test_df["is_declining_label"],
    baseline_pred
)

print("Baseline F1:", baseline_f1)
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

results = {
    "Accuracy": accuracy_score(y_test,pred),
    "Precision": precision_score(y_test,pred),
    "Recall": recall_score(y_test,pred),
    "F1": f1_score(y_test,pred)
}

results


| Model | Accuracy | Precision | Recall | F1 |
|---------|----------|-----------|--------|------|
| Week-4 Baseline Rule | XX | XX | XX | XX |
| LightGBM | XX | XX | XX | XX |

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Error Analysis

The model performs well on strongly declining pages but struggles with borderline cases.

Observed patterns:

1. False Positives
   - Some pages were predicted as declining because they were old and had falling impressions.
   - However, these pages still maintained stable click-through rates.

2. False Negatives
   - Some pages experienced sudden traffic drops that were not reflected in historical features.
   - The model underestimated these cases.

Feature importance suggests that:
- days_since_last_update
- impressions_90d
- clicks_90d
- trend_direction
- content_age_days

were among the strongest signals.

The model appears to rely heavily on engagement and freshness indicators, which is reasonable for refresh prioritization.

These results should be used as decision-support rather than automatic publishing decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.